In [2]:
# ============================================
# CERTIFICADO DE DEPÓSITO A TÉRMINO (CDT)
# Valery Montes - 01240372023
# Gabriel Plata - 01240372047
# ============================================

!pip install -q gradio

import gradio as gr
import csv


# ============================================
# CLASE CDT
# ============================================

class CDT:

    def __init__(self, montoInicial, plazoMeses):
        self.montoInicial = montoInicial
        self.plazoMeses = plazoMeses
        self.tasaMensual = 0
        self.historialProyeccion = []

    def obtener_resumen(self):
        saldoFinal = self.historialProyeccion[-1][2]

        return {
            "montoInicial": self.montoInicial,
            "plazoMeses": self.plazoMeses,
            "tasaMensual": self.tasaMensual,
            "saldoFinal": saldoFinal
        }


# ============================================
# CLASE CALCULADOR CDT
# ============================================

class CalculadorCDT:

    def calcular_tasa(self, plazo):
        return 0.001695 * plazo + 0.0983

    def generar_proyeccion(self, cdt):

        cdt.tasaMensual = self.calcular_tasa(cdt.plazoMeses)

        saldo = cdt.montoInicial

        cdt.historialProyeccion = [
            [0, 0, saldo]
        ]

        for mes in range(1, cdt.plazoMeses + 1):

            interes = saldo * (cdt.tasaMensual / 100)
            saldo = saldo + interes

            cdt.historialProyeccion.append([
                mes,
                interes,
                saldo
            ])


# ============================================
# CLASE GESTOR VISTA CDT
# ============================================

class GestorVistaCDT:

    def __init__(self):
        self.cdtActual = None
        self.motor = CalculadorCDT()

    def iniciar_sistema(self):
        pass

    def solicitar_datos(self, montoInicial, plazoMeses):

        if montoInicial is None or plazoMeses is None:
            raise ValueError("Debes ingresar el monto y el plazo.")

        if montoInicial <= 0:
            raise ValueError("El monto debe ser mayor que 0.")

        if plazoMeses < 1:
            raise ValueError("El plazo debe ser mayor o igual a 1.")

        return montoInicial, int(plazoMeses)

    def mostrar_tabla_proyeccion(self):

        if self.cdtActual is None:
            return []

        return self.cdtActual.historialProyeccion


# ============================================
# BASE DE DATOS
# ============================================

cdts = []

gestor = GestorVistaCDT()


# ============================================
# FUNCIONES AUXILIARES
# ============================================

def actualizar_tabla():

    datos = []

    for i, cdt in enumerate(cdts):

        resumen = cdt.obtener_resumen()

        datos.append([
            i,
            resumen["montoInicial"],
            resumen["plazoMeses"],
            resumen["tasaMensual"],
            resumen["saldoFinal"]
        ])

    return datos


def indice_valido(indice):

    if indice is None:
        return None

    indice = int(indice)

    if indice < 0 or indice >= len(cdts):
        return None

    return indice


def exportar_csv():

    if not cdts:
        return None

    archivo = "cdt.csv"

    with open(archivo, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "ID",
            "Monto inicial",
            "Plazo meses",
            "Tasa mensual",
            "Saldo final"
        ])

        for i, cdt in enumerate(cdts):

            resumen = cdt.obtener_resumen()

            writer.writerow([
                i,
                resumen["montoInicial"],
                resumen["plazoMeses"],
                resumen["tasaMensual"],
                resumen["saldoFinal"]
            ])

    return archivo


# ============================================
# OPERACIONES (una por cada opción del menú)
# ============================================

def op_agregar(indice, monto, plazo):

    try:
        monto, plazo = gestor.solicitar_datos(monto, plazo)
    except ValueError as error:
        return str(error), [], None

    cdt = CDT(monto, plazo)
    gestor.motor.generar_proyeccion(cdt)

    cdts.append(cdt)
    gestor.cdtActual = cdt

    return "CDT agregado correctamente.", [], None


def op_ver(indice, monto, plazo):

    i = indice_valido(indice)

    if i is None:
        return "El registro no existe. Revisa el ID.", [], None

    gestor.cdtActual = cdts[i]

    return (
        f"Proyección del CDT {i}.",
        gestor.mostrar_tabla_proyeccion(),
        None
    )


def op_editar(indice, monto, plazo):

    i = indice_valido(indice)

    if i is None:
        return "El registro no existe. Revisa el ID.", [], None

    # Si se deja un campo vacío, se conserva el valor actual
    if monto is None:
        monto = cdts[i].montoInicial
    if plazo is None:
        plazo = cdts[i].plazoMeses

    try:
        monto, plazo = gestor.solicitar_datos(monto, plazo)
    except ValueError as error:
        return str(error), [], None

    cdt = CDT(monto, plazo)
    gestor.motor.generar_proyeccion(cdt)

    cdts[i] = cdt
    gestor.cdtActual = cdt

    return "CDT actualizado correctamente.", [], None


def op_eliminar(indice, monto, plazo):

    i = indice_valido(indice)

    if i is None:
        return "El registro no existe. Revisa el ID.", [], None

    cdts.pop(i)

    gestor.cdtActual = cdts[-1] if cdts else None

    return "CDT eliminado correctamente.", [], None


def op_descargar(indice, monto, plazo):

    archivo = exportar_csv()

    if archivo is None:
        return "No hay CDTs para exportar.", [], None

    return "CSV generado. Descárgalo abajo.", [], archivo


OPERACIONES = {
    "Agregar": op_agregar,
    "Ver / Imprimir proyección": op_ver,
    "Editar": op_editar,
    "Eliminar": op_eliminar,
    "Descargar CSV": op_descargar
}


# ============================================
# BOTÓN EJECUTAR
# ============================================

def ejecutar(opcion, indice, monto, plazo):

    if opcion not in OPERACIONES:
        return actualizar_tabla(), "Selecciona una opción.", [], None

    mensaje, proyeccion, archivo = OPERACIONES[opcion](indice, monto, plazo)

    return actualizar_tabla(), mensaje, proyeccion, archivo


# ============================================
# MOSTRAR / OCULTAR CAMPOS SEGÚN LA OPCIÓN
# ============================================

def cambiar_opcion(opcion):

    usa_id = opcion in ("Ver / Imprimir proyección", "Editar", "Eliminar")
    usa_datos = opcion in ("Agregar", "Editar")
    muestra_proy = opcion == "Ver / Imprimir proyección"
    muestra_archivo = opcion == "Descargar CSV"

    return (
        gr.update(visible=usa_id),
        gr.update(visible=usa_datos),
        gr.update(visible=usa_datos),
        gr.update(visible=muestra_proy, value=[]),
        gr.update(visible=muestra_archivo, value=None),
        ""
    )


# ============================================
# INTERFAZ GRADIO (una sola sección)
# ============================================

with gr.Blocks(title="Calculador CDT") as app:

    gr.Markdown("# Calculador CDT")

    # PRESENTACIÓN: tabla de CDTs
    tabla = gr.Dataframe(
        headers=[
            "ID",
            "Monto inicial",
            "Plazo",
            "Tasa mensual",
            "Saldo final"
        ],
        value=actualizar_tabla(),
        interactive=False
    )

    # MENÚ DESPLEGABLE DE OPCIONES
    opcion = gr.Dropdown(
        choices=list(OPERACIONES.keys()),
        value="Agregar",
        label="Opción"
    )

    # CAMPOS (se muestran según la opción elegida)
    indice = gr.Number(
        label="ID del CDT",
        precision=0,
        visible=False
    )

    monto = gr.Number(
        label="Monto inicial",
        visible=True
    )

    plazo = gr.Number(
        label="Plazo en meses",
        precision=0,
        visible=True
    )

    # BOTÓN EJECUTAR
    boton_ejecutar = gr.Button("Ejecutar", variant="primary")

    # RESULTADOS
    mensaje = gr.Markdown()

    tabla_proyeccion = gr.Dataframe(
        headers=["Mes", "Interés", "Saldo"],
        interactive=False,
        visible=False
    )

    archivo_csv = gr.File(
        label="Archivo CSV",
        visible=False
    )

    # EVENTOS
    opcion.change(
        cambiar_opcion,
        inputs=opcion,
        outputs=[
            indice,
            monto,
            plazo,
            tabla_proyeccion,
            archivo_csv,
            mensaje
        ]
    )

    boton_ejecutar.click(
        ejecutar,
        inputs=[opcion, indice, monto, plazo],
        outputs=[tabla, mensaje, tabla_proyeccion, archivo_csv]
    )


# ============================================
# EJECUTAR
app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8a901364108fcf1d5e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
